# AMPS I · Week 02 — HW2: Unisolvence and nodal bases
**Due:** before the Week 3 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (25 pts)
Use condition (3) of the unisolvence lemma: $N_i(v)=0\ \forall i \Rightarrow v\equiv0$.

**(a) (8)** $K=[0,1]$, $\mathscr P=\mathscr P_2$, $\mathscr N=\{p(0),\,p(1),\,\int_0^1p\,dx\}$. Prove unisolvence and give the nodal basis.

**(b) (10)** Prove that the degree-3 Lagrange element on a triangle is unisolvent (the base case $k=3$ of the induction).
Use the hyperplane lemma; state clearly which nodes you use in each step.

**(c) (7)** $K=[0,1]$, $\mathscr P=\mathscr P_3$, $\mathscr N=\{p(0),\,p'(0),\,p(1),\,p'(1)\}$ (1D cubic Hermite). Prove unisolvence.

> *your proofs*

## Q2 — Your own nodal basis, compared with Basix (25 pts)
**(a) (10)** Implement `monomials(pts, k)` (shape `(n_mono, n_pts)`, order $1,x,y,x^2,xy,y^2,\dots$) and
`nodal_basis(nodes, k)` returning $\mu=V^{-1}$ and $\operatorname{cond}(V)$ for point-evaluation DOFs.
Use the **Basix node coordinates** `e.points` of the equispaced P3 element as your nodes, so your basis has the Basix order.
Compare your basis with `e.tabulate` at the degree-12 quadrature points: report $\max|\phi^{\rm yours}-\phi^{\rm Basix}|$.

**(b) (8)** Save your P3 basis on a fine lattice and plot all 10 functions (2×5 grid, title = DOF number and entity type).

**(c) (7)** Repeat (a) for $k=1,\dots,12$, with equispaced **and** GLL-warped nodes. Tabulate $\operatorname{cond}(V)$ and the max difference.
At which degree does your code lose 6 digits? Why does Basix not have this problem?

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
# TODO: monomials(pts, k), nodal_basis(nodes, k)
# TODO: (a) compare with Basix P3 (equispaced) at degree-12 quadrature points
# TODO: (b) save your P3 basis on a lattice → /tmp/fnx/hw2_p3.npz (also save e.points and the entity type of each DOF)
# TODO: (c) table for k = 1..12, both variants

In [ ]:
# TODO: (b) load /tmp/fnx/hw2_p3.npz and plot the 10 basis functions (2×5)

> *(c) your answer*

## Q3 — When unisolvence fails (20 pts)
Take six points on the circle of radius $0.3$ centred at $(0.5,0.5)$: angles $\theta_i = 0.3 + i\pi/3$, $i=0,\dots,5$.
Use them as point-evaluation DOFs for $\mathscr P_2$.

**(a) (6)** Compute the singular values of the $6\times6$ monomial Vandermonde matrix. Is it invertible?

**(b) (8)** Find the null vector of $V^T$ (last right singular vector) and identify the polynomial $v=\sum_i c_i\psi_i$ it represents.
Show that $v$ vanishes at all six nodes. Which condition of the unisolvence lemma fails?

**(c) (6)** Move one node by $\varepsilon\in\{10^{-1},10^{-2},10^{-4},10^{-8}\}$ (in $x$). How does $\operatorname{cond}(V)$ scale with $\varepsilon$?
What does this mean for a mesh with a nearly degenerate node arrangement?

In [ ]:
import numpy as np
th = 0.3 + np.arange(6) * np.pi / 3
X, Y = 0.5 + 0.3 * np.cos(th), 0.5 + 0.3 * np.sin(th)
# TODO: (a) singular values, (b) null vector and the polynomial, (c) perturbation study

> *your answers to (b) and (c)*

## Q4 — LLM-assisted: the cubic Hermite triangle (30 pts)
Cubic Hermite on the reference triangle (lecture §A9): $\mathscr P_3$; DOFs = value and gradient $(\partial_x,\partial_y)$ at each vertex, value at the centroid.
Basix has it: `basix.create_element(basix.ElementFamily.Hermite, basix.CellType.triangle, 3)`.

**(a) (12)** Ask an LLM (with the API sheet) to write a **generalized Vandermonde** construction of the Hermite basis from monomials
(the DOFs include derivatives!). Paste the unchanged generated code in the Markdown cell, put your corrected version in the code cell, and verify:
(i) $N_i(\phi_j)=\delta_{ij}$ numerically; (ii) agreement with Basix's Hermite basis at quadrature points. *(Find Basix's DOF order from `entity_dofs` and the tabulated values/derivatives at the vertices.)*

**(b) (10)** Basix reports `sobolev_space` = `H2` for this element. Compute the **normal derivative** of the centroid basis function $\phi_9$ along each edge.
Then answer: if two triangles share an edge and agree in all vertex DOFs, is the normal derivative continuous across the edge?
Is the global cubic Hermite space $C^1$? *(Hint: what is $\phi_9$? Compare with $\lambda_0\lambda_1\lambda_2$.)*

**(c) (8)** Counting argument: along an edge, the normal derivative of a polynomial of degree $k$ is a 1D polynomial of degree $k-1$.
How many conditions fix it, and how many do (i) cubic Hermite and (ii) quintic Argyris supply from the DOFs **shared** by both triangles?

**(a) generated code (unchanged)**

```python
# paste here
```

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
# TODO: (a) generalized Vandermonde for cubic Hermite; check N_i(phi_j) = delta_ij; compare with Basix
# TODO: (b) normal derivative of phi_9 along the three edges

> *(b) and (c) your answers*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …